In [ ]:
import os
from pathlib import Path

import pandas as pd
import torch

In [ ]:
data_dir = Path("/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m")
csv_path = data_dir / "data.csv"

In [ ]:
patient_dirs = [dir for dir in os.listdir(data_dir) if dir.startswith("TCGA")]

for patient_dir in patient_dirs:
    patient_dir_path = data_dir / patient_dir
    
    for image_name in os.listdir(patient_dir_path):
        if "_mask" in image_name:
            continue
    
        mask_name = f"{image_name[:-4]}_mask.tif"
        mask_image_path = patient_dir_path / mask_name
        
        if not os.path.exists(mask_image_path):
            print(f"Mask for {image_name} doesn't exist")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

In [ ]:
class MRIDataset(Dataset):
    def __init__(self, path=data_dir, transform=None):
        self.images = []
        self.transform = transform
        
        patient_dirs = [dir for dir in os.listdir(path) if dir.startswith("TCGA")]

        for patient_dir in patient_dirs:
            patient_dir_path = data_dir / patient_dir
            
            for image_name in os.listdir(patient_dir_path):
                if "_mask" in image_name:
                    continue

                image_path = patient_dir_path / image_name
                self.images.append(image_path)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = str(self.images[idx])
        mask_path  = f"{image_path[:-4]}_mask.tif"

        image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(mask_path, cv2.COLOR_BGR2GRAY)[:,:,None]

        if self.transform:
            image = self.transform(image)
            mask  = self.transform(mask)
        
        return image, mask

In [ ]:
import cv2
import matplotlib.pyplot as plt

In [ ]:
transform = transforms.ToTensor()

mri_dataset = MRIDataset(transform=transform)
image, mask = mri_dataset[0]
image.shape, mask.shape

In [ ]:
nrows, ncols = 3, 2
fig, axes = plt.subplots(nrows, ncols, figsize=(4*nrows,15))

for row in range(nrows):
    image, mask = mri_dataset[row]
    axes[row][0].imshow(image.permute(1,2,0))
    axes[row][1].imshow(mask.permute(1,2,0), cmap="gray")

plt.show()